In [1]:
import torch
from torchvision.models import resnet50
import ipywidgets as widgets
from PIL import Image
import io
import scripts.train as train
from torchvision.transforms import v2

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
model = resnet50(weights=None)
model.fc = torch.nn.Linear(model.fc.in_features, 1)
model.load_state_dict(torch.load('model/model.pt'))
model = model.to(device)
model = model.eval()

In [119]:
_, transform = train.set_transforms()
display_transform = v2.Resize((224, 224))

In [125]:
btn_upload = widgets.FileUpload()
prediction_label = widgets.Label()
out_pl = widgets.Output(layout=widgets.Layout(
        width="224px",
        height="224px",
        border="1px solid #ccc",
    ))
btn_run = widgets.Button(description="Classify Scan")

In [ ]:
def on_upload_finish(change):
    if btn_upload.value:
        with out_pl:
            out_pl.clear_output()
            img = Image.open(io.BytesIO(btn_upload.value[0]['content']))
            display(display_transform(img))

In [127]:
def on_button_clicked(b):
    img = Image.open(io.BytesIO(btn_upload.value[0]['content']))
    img_tensor = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        output = torch.sigmoid(model(img_tensor)).item()
    if output > 0.5:
        prediction_label.value = f"Prediction: Tumour Detected with {output*100:.2f}% confidence."
    else:
        prediction_label.value = f"Prediction: No Tumour Detected with { (1 - output) * 100:.2f}% confidence."

In [128]:
btn_upload.observe(on_upload_finish, names='value')
btn_run.on_click(on_button_clicked)

In [129]:
widgets.VBox([widgets.Label('Select a brain scan to classify'), btn_upload, out_pl, btn_run, prediction_label])